# L2c: Arrays, Dictionaries, Tables, and Collection Operations

Representation should follow the operations a problem requires. We will move the same reactor-run data through vectors, dictionaries, named tuples, and a `DataFrame`.

> **Learning Objectives**
> 1. Select a collection based on access and transformation needs.
> 2. Distinguish mutating and non-mutating operations.
> 3. Build a table from named records.
> 4. Filter, transform, group, and summarize tabular data.

---

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

In [ ]:
include(joinpath(@__DIR__, "Include.jl"))

## Arrays: ordered numerical data

A vector is a good representation when order, indexing, and elementwise numerical operations matter.

In [ ]:
conversion = [0.71, 0.76, 0.81, 0.68, 0.78, 0.83]
conversion_percent = 100 .* conversion
(first_run = conversion[1], selected_runs = conversion[[2, 5]], percent = conversion_percent)

## Dictionaries and named records

A dictionary maps keys to values. A named tuple is a compact immutable record whose fields are known by name.

In [ ]:
units = Dict(:temperature => "K", :feed_rate => "mol/s", :conversion => "fraction")
records = [
    (run = 1, catalyst = "A", temperature_K = 330.0, feed_mol_s = 1.00, conversion = 0.71),
    (run = 2, catalyst = "A", temperature_K = 340.0, feed_mol_s = 1.00, conversion = 0.76),
    (run = 3, catalyst = "A", temperature_K = 350.0, feed_mol_s = 0.95, conversion = 0.81),
    (run = 4, catalyst = "B", temperature_K = 330.0, feed_mol_s = 1.05, conversion = 0.68),
    (run = 5, catalyst = "B", temperature_K = 340.0, feed_mol_s = 1.00, conversion = 0.78),
    (run = 6, catalyst = "B", temperature_K = 350.0, feed_mol_s = 1.05, conversion = 0.83),
]
(conversion_unit = units[:conversion], first_record = records[1])

## Tables: columns plus row relationships

A `DataFrame` is useful when records share a schema and we need column selection, row filtering, grouping, or joins.

In [ ]:
runs = DataFrame(records)
runs.product_mol_s = runs.feed_mol_s .* runs.conversion
runs

`filter` returns selected rows without changing `runs`. The predicate states the engineering acceptance rule directly.

In [ ]:
accepted_runs = filter(row -> row.conversion >= 0.75, runs)
accepted_runs

## Group and summarize

Grouping preserves the individual rows while defining which rows contribute to each aggregate.

In [ ]:
catalyst_summary = combine(
    groupby(runs, :catalyst),
    :conversion => mean => :mean_conversion,
    :product_mol_s => sum => :total_product_mol,
    nrow => :number_of_runs,
)
sort!(catalyst_summary, :catalyst)

The shared Week 2 function accepts any numerical vector and returns a named summary without mutating its input.

In [ ]:
conversion_report = measurement_summary(runs.conversion)

## Check representation and operations

In [ ]:

@testset "collections and tables" begin
    @test conversion_percent[3] == 81.0
    @test units[:temperature] == "K"
    @test nrow(runs) == 6
    @test nrow(accepted_runs) == 4
    @test names(runs) == ["run", "catalyst", "temperature_K", "feed_mol_s", "conversion", "product_mol_s"]
    @test catalyst_summary.number_of_runs == [3, 3]
    @test catalyst_summary.mean_conversion[1] ≈ 0.76
    @test conversion_report.count == 6
    @test conversion_report.minimum == 0.68
end

## Summary

> **Key Takeaways**
> 1. Arrays support ordered numerical operations; dictionaries support key lookup.
> 2. Named tuples represent records; tables organize many records with one schema.
> 3. Filtering, transformation, grouping, and aggregation should express the question being asked.

---